In [28]:
import torch
from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation
import numpy as np
import cv2
import warnings
warnings.filterwarnings("ignore", category=FutureWarning) # huggingface is mad

In [ ]:
input_video = 'data/test_1.mp4'
output_video = 'data/output_1.mp4'
TARGET_SIZE = 1024
SHIRT_LABEL = 4
BATCH_SIZE = 8

ALPHA = 0.35                  # mask transparency
MASK_COLOR = (0, 255, 0)      # (Blue, Green, Red)

In [30]:
model_name = "mattmdjaga/segformer_b2_clothes"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
processor = SegformerImageProcessor.from_pretrained("mattmdjaga/segformer_b2_clothes")
model = AutoModelForSemanticSegmentation.from_pretrained("mattmdjaga/segformer_b2_clothes").to(device) # move to gpu if available

model.eval()

print(f'Inference running on {device}')


Inference running on cuda


In [31]:
from video_segmentation import get_single_mask_from_video

shirt_tensor = get_single_mask_from_video(input_video, target_resolution=TARGET_SIZE, batch_size=BATCH_SIZE, target_label=SHIRT_LABEL, processor=processor, model=model)

In [32]:
shirt_tensor.shape

torch.Size([469, 1024, 1024])

In [33]:
# Open the original video
cap = cv2.VideoCapture(input_video)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_idx = 0

# Prepare video writer
ret, frame = cap.read()
if not ret:
    raise RuntimeError("Cannot read video")

height, width = frame.shape[:2]
writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)  # reset to first frame

# Iterate over frames
while cap.isOpened() and frame_idx < len(shirt_tensor):
    ret, frame = cap.read()
    if not ret:
        break

    # Resize mask back to original frame size
    mask = shirt_tensor[frame_idx].numpy().astype(np.uint8) * 255
    mask_resized = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)

    # Convert mask to 3 channels
    mask_bgr = np.zeros_like(frame)        # shape (H, W, 3)
    mask_bgr[mask_resized > 0] = MASK_COLOR

    # Overlay mask
    blended = cv2.addWeighted(frame, 1 - ALPHA, mask_bgr, ALPHA, 0)

    writer.write(blended)
    frame_idx += 1

cap.release()
writer.release()
print(f"Output saved to {output_video}")

Output saved to data/new_output_1.mp4


In [34]:
overlay_img = cv2.imread("data/noise.jpg")  # BGR

In [ ]:
# Code to overlay the pattern onto the original video

cap = cv2.VideoCapture(input_video)
fps = cap.get(cv2.CAP_PROP_FPS)

ret, first_frame = cap.read()
h, w = first_frame.shape[:2]

writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h),
)

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# Resize overlay image to video resolution
overlay_img = cv2.resize(overlay_img, (w, h), interpolation=cv2.INTER_LINEAR)

frame_idx = 0

while cap.isOpened() and frame_idx < shirt_tensor.shape[0]:
    ret, frame = cap.read()
    if not ret:
        break

    # Get mask for this frame
    mask = shirt_tensor[frame_idx].numpy().astype(np.uint8)
    mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    # Expand mask to 3 channels
    mask_3c = mask[:, :, None]  # shape (H, W, 1)

    # Compose overlay only where mask == 1
    composed = frame.copy()
    composed[mask.astype(bool)] = overlay_img[mask.astype(bool)]

    writer.write(composed)
    frame_idx += 1

cap.release()
writer.release()